# ollama-local — the whole stack, on a model nobody bills you for

Cendor works with no cloud at all. The interesting part is the step that **cannot** be honest: a local model has no invoice.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## The five steps

Every recipe in `providers/` walks the same five, in the same order:

| # | Step | Here |
|---|---|---|
| 1 | **connect** | `ollama.Client()` — a local daemon, no cloud at all |
| 2 | **instrument** | one wrap — detection is structural, not name-based |
| 3 | **govern** | a `tokenguard` budget **and** a `guardrails` gate |
| 4 | **record** | `cassette` — the same call replayed offline, 0 provider calls |
| 5 | **prove** | `acttrace` `verify()` and a cost that came from `prices` |

**Distinctive here: the cost step documents its omission instead of faking a number.** `llama3` carries a $0.00 row; `llama3.2:latest` carries none at all and `call.cost` is `None`. So cap **tokens**, which needs no rate.

## 1–2 · Connect and instrument

In [ ]:
import main as recipe
from cendor.core import bus
from cendor.core.types import LLMCall

seen = []
bus.subscribe(lambda e: seen.append(e) if isinstance(e, LLMCall) else None)
client = recipe.make_client()  # OLLAMA_LIVE=1 swaps in the real daemon
client.chat(model=recipe.MODEL, messages=[{"role": "user", "content": "summarize"}])
call = seen[-1]
print(f"usage: {call.usage.input_tokens} in + {call.usage.output_tokens} out")
print(f"cost : {call.cost}")

## 3 · The cap that works with no rate

In [ ]:
from cendor.tokenguard import BudgetExceeded, budget

try:
    with budget(tokens=100, on_exceed="block"):
        client.chat(model=recipe.MODEL, messages=[{"role": "user", "content": "summarize again"}])
except BudgetExceeded as e:
    refusal = str(e).splitlines()[0]
    print(refusal)

## 5 · Prove it

In [ ]:
assert call.usage.input_tokens > 0, "prompt_eval_count was not normalized"
assert refusal, "a token cap must bind even with no rate for the model"
print("OK")